**Autor:** Andrés Alejandro Rodríguez Lozano  
**Portafolio:** Portafolio de Andrés Alejandro Rodríguez Lozano

# Capítulo 4 — Métricas de riesgo/retorno

$$\mathrm{CAGR}=(V_T/V_0)^{1/T}-1,\quad \sigma=\mathrm{std}(R_m)\sqrt{12}$$


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%config InlineBackend.figure_formats = ['png']
%config InlineBackend.figure_format = 'png'
plt.rcParams.update({
    'figure.dpi': 72,
    'savefig.dpi': 72,
    'figure.figsize': (9, 4.2),
    'font.size': 9,
    'svg.fonttype': 'none',
    'path.simplify': True,
})

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import portfolio_utils as pu

daily = pd.read_csv(ROOT/'data'/'precios_ajustados_diarios.csv', index_col=0, parse_dates=True)
monthly = pu.to_month_end(daily).loc[: '2026-09-30']
bt = pu.backtest_portfolio(monthly, pu.CURRENT_WEIGHTS, start='2016-01-31', end='2026-09-30', rebalance='A')
spy = pu.backtest_portfolio(monthly, {'SPY':1}, start='2016-01-31', end='2026-09-30', rebalance='N')
rf = pu.load_risk_free_monthly()
m_p = pu.series_metrics(bt['value'], spy['value'], rf, label='Portafolio')
m_s = pu.series_metrics(spy['value'], None, rf, label='SPY')
print(f'{"Métrica":22s} {"Portafolio":>12s} {"SPY":>12s}')
rows = [
    ('CAGR', 'cagr', True),
    ('Volatilidad', 'stdev', True),
    ('Max DD', 'max_drawdown', True),
    ('Sharpe', 'sharpe', False),
    ('Sortino', 'sortino', False),
    ('Valor final', 'end_balance', False),
]
for lab, k, pct in rows:
    vp, vs = m_p[k], m_s[k]
    if pct:
        print(f'{lab:22s} {100*vp:11.2f}% {100*vs:11.2f}%')
    elif k == 'end_balance':
        print(f'{lab:22s} {vp:12.2f} {vs:12.2f}')
    else:
        print(f'{lab:22s} {vp:12.2f} {vs:12.2f}')
for k in ['beta','alpha','upside_capture','downside_capture','tracking_error','information_ratio']:
    if k in m_p and m_p[k] == m_p[k]:  # not NaN
        print(f'{k:22s} {m_p[k]:.4f}')


Métrica                  Portafolio          SPY
CAGR                         19.21%       14.89%
Volatilidad                  16.54%       15.04%
Max DD                      -27.47%      -23.93%
Sharpe                         1.02         0.85
Sortino                        0.93         0.77
Valor final                66141.36     44481.57
beta                   1.0322
alpha                  0.0362
upside_capture         108.7956
downside_capture       89.5552
tracking_error         0.0573
information_ratio      0.7542


**Interpretación:** mayor CAGR y Sharpe vs SPY a costa de Max DD más profundo.

## Conclusiones
1. El portafolio supera a SPY en retorno ajustado por riesgo en 2016–2026, con peores caídas.

## Ejercicios
1. Sharpe con rf=0.
2. Sortino vs Sharpe en 2022.
3. Calmar = CAGR/|Max DD|.
